// ==================================================================================================
// NOTEBOOK 10 — CONTROLE DE L'ENTRAINEMENT
// ==================================================================================================
//
//  EarlyStopping : Arret automatique lorsque val_loss stagne  (patience 3 / 7 / 15)
//  GridSearchCV  : Recherche exhaustive des hyperparametres ML  (Random Forest + XGBoost)
//  Optuna        : Optimisation bayesienne TPE des hyperparametres ML
//
//  Modeles DL : MLP (128->64->32->1) — donnees .npy (notebooks 05/06)
//               LSTM (64 units)      — featured_data.parquet (notebook 09)
//  Modeles ML : Random Forest | XGBoost — featured_data.parquet (notebook 04)
//  Target     : taxi_out (minutes)
//  Metriques  : MAE | RMSE | R2
//
// ==================================================================================================

// IMPORTS
// pip install optuna  (si non installe)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
import time
warnings.filterwarnings('ignore')

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler

# Machine Learning
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

# Optuna
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("TensorFlow :", tf.__version__)
print("Optuna     :", optuna.__version__)

// ==================================================================================================
// SECTION 0 — CHARGEMENT DES DONNEES
// ==================================================================================================
//  DL  : fichiers .npy produits par notebook 05 (70 / 15 / 15 %)
//  ML  : featured_data.parquet produit par notebook 03
//  ML* : sous-echantillon 50 000 lignes pour GridSearch / Optuna (performance)
// ==================================================================================================

In [ ]:
# -- DL data (X_train / X_val / X_test + y) --
X_train_dl = np.load("../data/X_train.npy")
X_val_dl   = np.load("../data/X_val.npy")
X_test_dl  = np.load("../data/X_test.npy")
y_train_dl = np.load("../data/y_train.npy")
y_val_dl   = np.load("../data/y_val.npy")
y_test_dl  = np.load("../data/y_test.npy")

print("DL Train :", X_train_dl.shape,
      "| Val :", X_val_dl.shape,
      "| Test :", X_test_dl.shape)

In [ ]:
# -- ML data (featured_data.parquet) --
df_ml = pd.read_parquet("../data/featured_data.parquet")

y_ml = df_ml["taxi_out"]
X_ml = df_ml.drop(columns=["taxi_out"])
X_ml = pd.get_dummies(X_ml, drop_first=True).fillna(0)

# Sous-echantillon 50 000 lignes pour GridSearch / Optuna
SAMPLE = 50_000
np.random.seed(42)
idx  = np.random.choice(len(X_ml), SAMPLE, replace=False)
X_s, y_s = X_ml.iloc[idx], y_ml.iloc[idx]

X_tr_ml, X_te_ml, y_tr_ml, y_te_ml = train_test_split(
    X_s, y_s, test_size=0.2, random_state=42
)

print(f"ML Train : {X_tr_ml.shape}  |  Test : {X_te_ml.shape}")
print(f"Features : {X_tr_ml.shape[1]}")

// ==================================================================================================
// SECTION 1 — EARLYSTOPPING : THEORIE
// ==================================================================================================
//
//  Principe : surveiller une metrique de validation pendant l'entrainement.
//  Si la metrique ne s'ameliore pas pendant `patience` epochs consecutives → arret.
//
//  Parametres cles (Keras) :
//    monitor              = "val_loss"   metrique surveillee
//    patience             = 5            epochs sans amelioration toleres
//    restore_best_weights = True         restaurer les poids du meilleur epoch
//    min_delta            = 0.001        amelioration minimale considered
//    mode                 = "min"        on minimise val_loss
//
//  Risques :
//    patience trop faible  → arret premature (underfitting)
//    patience trop haute   → gaspillage de temps, risque d'overfitting leger
//
//  Bonne pratique : combiner avec ReduceLROnPlateau
//    LRPlateau  : reduit le learning rate si val_loss stagne (patience_lr < patience_es)
//    EarlyStopping : arrete si plus aucune amelioration possible
//
// ==================================================================================================

In [ ]:
print("GUIDE PRATIQUE — CHOIX DE LA PATIENCE")
print("=" * 58)
print(f"{'Patience':<12} {'Risque':<22} {'Usage typique'}")
print("-" * 58)
rows = [
    ("1 - 3",   "Arret premature",     "Prototype / Debug rapide"),
    ("5 - 7",   "Bon equilibre",       "Production standard"),
    ("10 - 15", "Entrainement long",   "Dataset difficile / bruye"),
    ("20 +",    "Quasi-inutile",       "Convergence tres lente"),
]
for p, r, u in rows:
    print(f"  {p:<10} {r:<22} {u}")
print("=" * 58)
print()
print("Exemple combine :")
print("  EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)")
print("  ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)")

// ==================================================================================================
// SECTION 2 — EARLYSTOPPING SUR MLP : COMPARAISON patience = 3 / 7 / 15
// ==================================================================================================
//  Architecture : Input(27) → Dense(128,relu) → Dropout(0.3) → Dense(64,relu)
//                           → Dropout(0.3) → Dense(32,relu) → Dropout(0.2) → Dense(1)
//  Optimizer    : Adam  |  Loss : MSE  |  Metrics : MAE
//  Epochs max   : 100   |  Batch size  : 256  (plus rapide sur gros dataset)
//  Donnees      : X_train/val/test.npy  (237280 / 50846 / 50846)
// ==================================================================================================

In [ ]:
def build_mlp():
    model = Sequential([
        Input(shape=(X_train_dl.shape[1],)),
        Dense(128, activation="relu"),
        Dropout(0.3),
        Dense(64, activation="relu"),
        Dropout(0.3),
        Dense(32, activation="relu"),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

model_test = build_mlp()
model_test.summary()

In [ ]:
results_es_mlp = {}

for patience in [3, 7, 15]:
    print(f"\n{'='*45}")
    print(f"  MLP — EarlyStopping  patience = {patience}")
    print(f"{'='*45}")

    model = build_mlp()

    es = EarlyStopping(
        monitor              = "val_loss",
        patience             = patience,
        restore_best_weights = True,
        min_delta            = 0.001,
        verbose              = 0
    )
    lr_sch = ReduceLROnPlateau(
        monitor  = "val_loss",
        factor   = 0.5,
        patience = max(2, patience // 2),
        verbose  = 0
    )

    t0 = time.time()
    hist = model.fit(
        X_train_dl, y_train_dl,
        validation_data = (X_val_dl, y_val_dl),
        epochs          = 100,
        batch_size      = 256,
        callbacks       = [es, lr_sch],
        verbose         = 0
    )
    elapsed = time.time() - t0

    y_pred = model.predict(X_test_dl, verbose=0).flatten()
    mae    = mean_absolute_error(y_test_dl, y_pred)
    rmse   = np.sqrt(mean_squared_error(y_test_dl, y_pred))
    vl     = hist.history["val_loss"]
    be     = vl.index(min(vl))

    results_es_mlp[patience] = {
        "history" : hist.history,
        "mae"     : mae,
        "rmse"    : rmse,
        "epochs"  : len(vl),
        "best_ep" : be,
        "time"    : elapsed
    }

    print(f"  Epochs entraines  : {len(vl)}")
    print(f"  Meilleur epoch    : {be + 1}")
    print(f"  MAE  (test)       : {mae:.4f} min")
    print(f"  RMSE (test)       : {rmse:.4f} min")
    print(f"  Temps             : {elapsed:.1f}s")

In [ ]:
# -- Courbes d'apprentissage avec point d'arret --
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
pal = {3: "steelblue", 7: "seagreen", 15: "darkorange"}

for i, p in enumerate([3, 7, 15]):
    r  = results_es_mlp[p]
    vl = r["history"]["val_loss"]
    tl = r["history"]["loss"]
    be = r["best_ep"]
    ax = axes[i]

    ax.plot(tl, color=pal[p], alpha=0.85, linewidth=1.5, label="Train Loss")
    ax.plot(vl, color=pal[p], linestyle="--", linewidth=1.5, label="Val Loss")
    ax.axvline(be, color="red", linestyle=":",
               linewidth=2, label=f"Meilleur (ep {be+1})")
    ax.axvline(len(vl) - 1, color="black", linestyle="-.",
               linewidth=1.5, label=f"Arret (ep {len(vl)})")
    ax.set_title(
        f"patience = {p}\n"
        f"MAE = {r['mae']:.3f}  |  {r['epochs']} epochs  |  {r['time']:.0f}s",
        fontsize=10
    )
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss (MSE)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)

plt.suptitle("EarlyStopping — MLP TaxiOut : Comparaison des patience", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# -- Tableau comparatif patience --
summary_mlp = pd.DataFrame({
    "Patience"       : [3, 7, 15],
    "Epochs"         : [results_es_mlp[p]["epochs"]    for p in [3, 7, 15]],
    "Meilleur epoch" : [results_es_mlp[p]["best_ep"]+1 for p in [3, 7, 15]],
    "MAE test"       : [round(results_es_mlp[p]["mae"],  4) for p in [3, 7, 15]],
    "RMSE test"      : [round(results_es_mlp[p]["rmse"], 4) for p in [3, 7, 15]],
    "Temps (s)"      : [round(results_es_mlp[p]["time"], 1) for p in [3, 7, 15]]
})
print(summary_mlp.to_string(index=False))

best_p_mlp = min([3, 7, 15], key=lambda p: results_es_mlp[p]["mae"])
print(f"\n=> Meilleure patience MLP : {best_p_mlp}  "
      f"(MAE = {results_es_mlp[best_p_mlp]['mae']:.4f})")

# Bar chart MAE + RMSE
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(3)
w = 0.35
b1 = ax.bar(x - w/2, summary_mlp["MAE test"],  w,
            color="steelblue",  label="MAE",  edgecolor="black")
b2 = ax.bar(x + w/2, summary_mlp["RMSE test"], w,
            color="darkorange", label="RMSE", edgecolor="black", alpha=0.85)
for b, v in zip(b1, summary_mlp["MAE test"]):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.04,
            f"{v:.3f}", ha="center", fontsize=9)
for b, v in zip(b2, summary_mlp["RMSE test"]):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.04,
            f"{v:.3f}", ha="center", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(["patience=3", "patience=7", "patience=15"])
ax.set_ylabel("Erreur (minutes)")
ax.set_title("MAE vs RMSE selon la patience EarlyStopping (MLP)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

// ==================================================================================================
// SECTION 3 — EARLYSTOPPING SUR LSTM
// ==================================================================================================
//  Architecture : Input(10, 6) → LSTM(64) → Dropout(0.3) → Dense(32, relu) → Dense(1)
//  Donnees      : featured_data.parquet triees chronologiquement (notebook 09)
//  Sequences    : longueur 10  |  Target : departure_delay (normalise)
//  Comparaison  : patience = 3 / 5 / 10
// ==================================================================================================

In [ ]:
# -- Preparation des sequences LSTM (identique notebook 09) --
df_lstm = pd.read_parquet("../data/featured_data.parquet").dropna()
df_lstm["date"] = pd.to_datetime(df_lstm["date"])
df_lstm = df_lstm.sort_values("date").reset_index(drop=True)

features_lstm = ["departure_delay", "scheduled_elapsed_time", "departure_hour",
                 "airport_traffic", "carrier_traffic", "is_peak_hour"]
features_lstm = [c for c in features_lstm if c in df_lstm.columns]

scaler_lstm = StandardScaler()
scaled_lstm = scaler_lstm.fit_transform(df_lstm[features_lstm])

SEQ_LEN = 10
X_seq, y_seq = [], []
for i in range(SEQ_LEN, len(scaled_lstm)):
    X_seq.append(scaled_lstm[i - SEQ_LEN:i])
    y_seq.append(scaled_lstm[i][0])
X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

n_tr  = int(0.70 * len(X_seq))
n_val = int(0.15 * len(X_seq))
Xl_tr,  yl_tr  = X_seq[:n_tr],            y_seq[:n_tr]
Xl_val, yl_val = X_seq[n_tr:n_tr+n_val],  y_seq[n_tr:n_tr+n_val]
Xl_te,  yl_te  = X_seq[n_tr+n_val:],      y_seq[n_tr+n_val:]

print("LSTM sequences :", X_seq.shape)
print("Train :", Xl_tr.shape, "| Val :", Xl_val.shape, "| Test :", Xl_te.shape)

In [ ]:
def build_lstm():
    m = Sequential([
        Input(shape=(SEQ_LEN, len(features_lstm))),
        LSTM(64, return_sequences=False),
        Dropout(0.3),
        Dense(32, activation="relu"),
        Dense(1)
    ])
    m.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return m

results_es_lstm = {}

for patience in [3, 5, 10]:
    print(f"\n--- LSTM  patience = {patience} ---")
    m  = build_lstm()
    es = EarlyStopping(monitor="val_loss", patience=patience,
                       restore_best_weights=True, verbose=0)
    lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                           patience=max(2, patience//2), verbose=0)
    t0 = time.time()
    hist = m.fit(
        Xl_tr, yl_tr,
        validation_data = (Xl_val, yl_val),
        epochs          = 30,
        batch_size      = 64,
        callbacks       = [es, lr],
        verbose         = 0
    )
    elapsed = time.time() - t0
    yp   = m.predict(Xl_te, verbose=0).flatten()
    mae  = mean_absolute_error(yl_te, yp)
    rmse = np.sqrt(mean_squared_error(yl_te, yp))
    r2   = r2_score(yl_te, yp)
    vl   = hist.history["val_loss"]
    be   = vl.index(min(vl))

    results_es_lstm[patience] = {
        "history": hist.history, "mae": mae, "rmse": rmse,
        "r2": r2, "epochs": len(vl), "best_ep": be, "time": elapsed
    }
    print(f"  Epochs : {len(vl)} | Best ep : {be+1} | MAE : {mae:.4f} | {elapsed:.0f}s")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
pal_lstm = {3: "steelblue", 5: "seagreen", 10: "darkorange"}

for i, p in enumerate([3, 5, 10]):
    r  = results_es_lstm[p]
    vl = r["history"]["val_loss"]
    tl = r["history"]["loss"]
    ax = axes[i]

    ax.plot(tl, color=pal_lstm[p], alpha=0.85, linewidth=1.5, label="Train Loss")
    ax.plot(vl, color=pal_lstm[p], linestyle="--", linewidth=1.5, label="Val Loss")
    ax.axvline(r["best_ep"], color="red", linestyle=":", linewidth=2,
               label=f"Best (ep {r['best_ep']+1})")
    ax.axvline(r["epochs"]-1, color="black", linestyle="-.", linewidth=1.5,
               label=f"Stop (ep {r['epochs']})")
    ax.set_title(f"LSTM patience={p}\nMAE={r['mae']:.4f} | R2={r['r2']:.4f}", fontsize=10)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss (MSE)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)

plt.suptitle("EarlyStopping — LSTM (TaxiOut) : Comparaison des patience", fontsize=13)
plt.tight_layout()
plt.show()

print("\nTableau recapitulatif LSTM :")
df_lstm_sum = pd.DataFrame([
    {"Patience": p, "Epochs": results_es_lstm[p]["epochs"],
     "Best ep": results_es_lstm[p]["best_ep"]+1,
     "MAE": round(results_es_lstm[p]["mae"], 4),
     "RMSE": round(results_es_lstm[p]["rmse"], 4),
     "R2": round(results_es_lstm[p]["r2"], 4)}
    for p in [3, 5, 10]
])
print(df_lstm_sum.to_string(index=False))
best_p_lstm = min([3, 5, 10], key=lambda p: results_es_lstm[p]["mae"])
print(f"\n=> Meilleure patience LSTM : {best_p_lstm}")

// ==================================================================================================
// SECTION 4 — GRIDSEARCHCV : RECHERCHE EXHAUSTIVE DES HYPERPARAMETRES
// ==================================================================================================
//
//  Principe : tester TOUTES les combinaisons d'une grille predefined.
//  Avantages  : simple, reproductible, garantit le meilleur dans la grille.
//  Inconvenients : cout O(N_combinaisons x cv_folds) — exponentiel.
//
//  sklearn.model_selection.GridSearchCV
//    param_grid = dictionnaire des valeurs a tester
//    scoring    = "neg_mean_absolute_error"
//    cv         = 3  (k-fold cross-validation)
//    n_jobs     = -1 (parallele sur tous les CPU disponibles)
//
//  Donnees : sous-echantillon 50 000 lignes  (40k train / 10k test)
//
// ==================================================================================================

In [ ]:
# -- Baselines notebook 04 (67 803 samples de test) --
RF_BASE_MAE,  RF_BASE_RMSE,  RF_BASE_R2  = 5.31, 8.98, 0.446
XGB_BASE_MAE, XGB_BASE_RMSE, XGB_BASE_R2 = 5.84, 9.70, 0.352

# ── GridSearchCV — Random Forest ────────────────────────────────────────
print("GridSearchCV — Random Forest")
print("=" * 55)

param_grid_rf = {
    "n_estimators"     : [100, 200],
    "max_depth"        : [10, 20, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf" : [1, 2],
}

total_rf = (
    len(param_grid_rf["n_estimators"]) *
    len(param_grid_rf["max_depth"]) *
    len(param_grid_rf["min_samples_split"]) *
    len(param_grid_rf["min_samples_leaf"]) * 3
)
print(f"Grille : {total_rf} fits  (CV=3)")

grid_rf = GridSearchCV(
    estimator  = RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid = param_grid_rf,
    scoring    = "neg_mean_absolute_error",
    cv         = 3,
    verbose    = 1,
    n_jobs     = -1
)
t0 = time.time()
grid_rf.fit(X_tr_ml, y_tr_ml)
grid_rf_time = time.time() - t0

print(f"\nTermine en {grid_rf_time:.1f}s")
print(f"Meilleurs params  : {grid_rf.best_params_}")
print(f"Meilleur CV-MAE   : {-grid_rf.best_score_:.4f}")

rf_gs_pred = grid_rf.best_estimator_.predict(X_te_ml)
rf_gs_mae  = mean_absolute_error(y_te_ml, rf_gs_pred)
rf_gs_rmse = np.sqrt(mean_squared_error(y_te_ml, rf_gs_pred))
rf_gs_r2   = r2_score(y_te_ml, rf_gs_pred)

print(f"Test MAE  : {rf_gs_mae:.4f}  (baseline {RF_BASE_MAE})")
print(f"Test RMSE : {rf_gs_rmse:.4f}  (baseline {RF_BASE_RMSE})")
print(f"Test R2   : {rf_gs_r2:.4f}   (baseline {RF_BASE_R2})")

In [ ]:
# -- Top 10 configurations RF --
cv_rf = pd.DataFrame(grid_rf.cv_results_)
cv_rf["CV_MAE"] = -cv_rf["mean_test_score"]
cv_rf_top = cv_rf.sort_values("CV_MAE").head(10).reset_index(drop=True)

cols_rf = ["param_n_estimators", "param_max_depth",
           "param_min_samples_split", "param_min_samples_leaf", "CV_MAE"]
print("Top 10 configurations RF (CV):")
print(cv_rf_top[cols_rf].to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
labels = [f"#{i+1}" for i in range(len(cv_rf_top))]
bars = ax.bar(labels, cv_rf_top["CV_MAE"], color="steelblue", edgecolor="black")
bars[0].set_color("gold")
ax.axhline(RF_BASE_MAE, color="red", linestyle="--",
           label=f"Baseline (NB04) = {RF_BASE_MAE}")
for b, v in zip(bars, cv_rf_top["CV_MAE"]):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01,
            f"{v:.3f}", ha="center", fontsize=8)
ax.set_title("GridSearchCV RF — Top 10 configurations par CV-MAE\n(or = meilleur)")
ax.set_ylabel("MAE (CV, minutes)")
ax.set_xlabel("Rang")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── GridSearchCV — XGBoost ──────────────────────────────────────────────
print("GridSearchCV — XGBoost")
print("=" * 55)

param_grid_xgb = {
    "n_estimators" : [100, 200],
    "max_depth"    : [4, 6, 8],
    "learning_rate": [0.05, 0.1, 0.2],
    "subsample"    : [0.8, 1.0],
}

total_xgb = (
    len(param_grid_xgb["n_estimators"]) *
    len(param_grid_xgb["max_depth"]) *
    len(param_grid_xgb["learning_rate"]) *
    len(param_grid_xgb["subsample"]) * 3
)
print(f"Grille : {total_xgb} fits  (CV=3)")

grid_xgb = GridSearchCV(
    estimator  = XGBRegressor(random_state=42, n_jobs=-1),
    param_grid = param_grid_xgb,
    scoring    = "neg_mean_absolute_error",
    cv         = 3,
    verbose    = 1,
    n_jobs     = -1
)
t0 = time.time()
grid_xgb.fit(X_tr_ml, y_tr_ml)
grid_xgb_time = time.time() - t0

print(f"\nTermine en {grid_xgb_time:.1f}s")
print(f"Meilleurs params  : {grid_xgb.best_params_}")
print(f"Meilleur CV-MAE   : {-grid_xgb.best_score_:.4f}")

xgb_gs_pred = grid_xgb.best_estimator_.predict(X_te_ml)
xgb_gs_mae  = mean_absolute_error(y_te_ml, xgb_gs_pred)
xgb_gs_rmse = np.sqrt(mean_squared_error(y_te_ml, xgb_gs_pred))
xgb_gs_r2   = r2_score(y_te_ml, xgb_gs_pred)

print(f"Test MAE  : {xgb_gs_mae:.4f}  (baseline {XGB_BASE_MAE})")
print(f"Test RMSE : {xgb_gs_rmse:.4f}  (baseline {XGB_BASE_RMSE})")
print(f"Test R2   : {xgb_gs_r2:.4f}   (baseline {XGB_BASE_R2})")

In [ ]:
# -- Top 10 configurations XGB --
cv_xgb = pd.DataFrame(grid_xgb.cv_results_)
cv_xgb["CV_MAE"] = -cv_xgb["mean_test_score"]
cv_xgb_top = cv_xgb.sort_values("CV_MAE").head(10).reset_index(drop=True)

cols_xgb = ["param_n_estimators","param_max_depth","param_learning_rate",
            "param_subsample","CV_MAE"]
print("Top 10 configurations XGBoost (CV):")
print(cv_xgb_top[cols_xgb].to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
labels = [f"#{i+1}" for i in range(len(cv_xgb_top))]
bars = ax.bar(labels, cv_xgb_top["CV_MAE"], color="seagreen", edgecolor="black")
bars[0].set_color("gold")
ax.axhline(XGB_BASE_MAE, color="red", linestyle="--",
           label=f"Baseline (NB04) = {XGB_BASE_MAE}")
for b, v in zip(bars, cv_xgb_top["CV_MAE"]):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01,
            f"{v:.3f}", ha="center", fontsize=8)
ax.set_title("GridSearchCV XGBoost — Top 10 configurations par CV-MAE")
ax.set_ylabel("MAE (CV, minutes)")
ax.set_xlabel("Rang")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

// ==================================================================================================
// SECTION 5 — OPTUNA : OPTIMISATION BAYESIENNE (TPE)
// ==================================================================================================
//
//  Principe : au lieu de tester TOUTES les combinaisons, Optuna guide la recherche
//  vers les regions prometteuses via l'algorithme TPE (Tree-structured Parzen Estimator).
//
//  Avantages vs GridSearchCV :
//    + Espace de recherche continu (float, int) — pas de grille discrete
//    + Convergence rapide : ~50 trials = souvent mieux que 100+ fits GridSearch
//    + Pruning : arreter les mauvais trials tot (callbacks Keras)
//    + Visualisation : historique, importance des parametres, parallel coordinates
//
//  API Optuna :
//    trial.suggest_int(name, low, high)
//    trial.suggest_float(name, low, high, log=True)   ← echelle log pour lr
//    trial.suggest_categorical(name, choices)
//    study.optimize(objective, n_trials=N)
//    study.trials_dataframe()
//
//  N_TRIALS = 50  —  ajuster selon le temps disponible
//
// ==================================================================================================

In [ ]:
N_TRIALS = 50

# ── Optuna — Random Forest ───────────────────────────────────────────────
def objective_rf(trial):
    params = {
        "n_estimators"     : trial.suggest_int("n_estimators", 50, 400),
        "max_depth"        : trial.suggest_int("max_depth", 5, 40),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 15),
        "min_samples_leaf" : trial.suggest_int("min_samples_leaf", 1, 6),
        "max_features"     : trial.suggest_categorical(
                                 "max_features", ["sqrt", "log2"]),
    }
    m = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
    m.fit(X_tr_ml, y_tr_ml)
    return mean_absolute_error(y_te_ml, m.predict(X_te_ml))

study_rf = optuna.create_study(
    direction  = "minimize",
    study_name = "RF_TaxiOut",
    sampler    = optuna.samplers.TPESampler(seed=42)
)

print(f"Optuna RF — {N_TRIALS} trials")
t0 = time.time()
study_rf.optimize(objective_rf, n_trials=N_TRIALS, show_progress_bar=True)
optuna_rf_time = time.time() - t0

print(f"\nTermine en {optuna_rf_time:.1f}s")
print(f"Meilleurs params : {study_rf.best_params}")
print(f"Meilleur MAE     : {study_rf.best_value:.4f}")

# Refit
rf_opt = RandomForestRegressor(**study_rf.best_params, random_state=42, n_jobs=-1)
rf_opt.fit(X_tr_ml, y_tr_ml)
rf_opt_pred = rf_opt.predict(X_te_ml)
rf_opt_mae  = mean_absolute_error(y_te_ml, rf_opt_pred)
rf_opt_rmse = np.sqrt(mean_squared_error(y_te_ml, rf_opt_pred))
rf_opt_r2   = r2_score(y_te_ml, rf_opt_pred)
print(f"\nTest MAE  : {rf_opt_mae:.4f}  (baseline {RF_BASE_MAE})")
print(f"Test RMSE : {rf_opt_rmse:.4f}")
print(f"Test R2   : {rf_opt_r2:.4f}")

In [ ]:
# -- Visualisation Optuna RF --
trials_rf  = [t.value for t in study_rf.trials]
best_rf_cm = np.minimum.accumulate(trials_rf)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Historique d'optimisation
axes[0].scatter(range(len(trials_rf)), trials_rf,
                alpha=0.5, color="steelblue", s=25, label="MAE trial")
axes[0].plot(range(len(best_rf_cm)), best_rf_cm,
             color="red", linewidth=2, label="Meilleur cumulatif")
axes[0].axhline(study_rf.best_value, color="orange", linestyle="--", alpha=0.7,
                label=f"Best = {study_rf.best_value:.3f}")
axes[0].set_title("Optuna RF — Historique d'optimisation")
axes[0].set_xlabel("Trial #")
axes[0].set_ylabel("MAE (minutes)")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Importance des hyperparametres (correlation |r| avec MAE)
df_t_rf = study_rf.trials_dataframe()
param_cols = [c for c in df_t_rf.columns if c.startswith("params_")]
corrs = {}
for col in param_cols:
    try:
        c = df_t_rf[col].astype(float).corr(df_t_rf["value"])
        corrs[col.replace("params_", "")] = abs(c) if not np.isnan(c) else 0
    except Exception:
        pass
corrs_s = dict(sorted(corrs.items(), key=lambda x: x[1], reverse=True))
axes[1].barh(list(corrs_s.keys()), list(corrs_s.values()),
             color="steelblue", edgecolor="black")
axes[1].set_title("RF — Importance des hyperparametres (|corr| avec MAE)")
axes[1].set_xlabel("|Correlation|")
axes[1].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Optuna — XGBoost ────────────────────────────────────────────────────
def objective_xgb(trial):
    params = {
        "n_estimators"    : trial.suggest_int("n_estimators", 50, 400),
        "max_depth"       : trial.suggest_int("max_depth", 3, 12),
        "learning_rate"   : trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample"       : trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma"           : trial.suggest_float("gamma", 0.0, 1.0),
        "reg_alpha"       : trial.suggest_float("reg_alpha", 1e-5, 1.0, log=True),
        "reg_lambda"      : trial.suggest_float("reg_lambda", 1e-5, 1.0, log=True),
    }
    m = XGBRegressor(**params, random_state=42, n_jobs=-1)
    m.fit(X_tr_ml, y_tr_ml, verbose=False)
    return mean_absolute_error(y_te_ml, m.predict(X_te_ml))

study_xgb = optuna.create_study(
    direction  = "minimize",
    study_name = "XGB_TaxiOut",
    sampler    = optuna.samplers.TPESampler(seed=42)
)

print(f"Optuna XGB — {N_TRIALS} trials")
t0 = time.time()
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS, show_progress_bar=True)
optuna_xgb_time = time.time() - t0

print(f"\nTermine en {optuna_xgb_time:.1f}s")
print(f"Meilleurs params : {study_xgb.best_params}")
print(f"Meilleur MAE     : {study_xgb.best_value:.4f}")

xgb_opt = XGBRegressor(**study_xgb.best_params, random_state=42, n_jobs=-1)
xgb_opt.fit(X_tr_ml, y_tr_ml, verbose=False)
xgb_opt_pred = xgb_opt.predict(X_te_ml)
xgb_opt_mae  = mean_absolute_error(y_te_ml, xgb_opt_pred)
xgb_opt_rmse = np.sqrt(mean_squared_error(y_te_ml, xgb_opt_pred))
xgb_opt_r2   = r2_score(y_te_ml, xgb_opt_pred)
print(f"\nTest MAE  : {xgb_opt_mae:.4f}  (baseline {XGB_BASE_MAE})")
print(f"Test RMSE : {xgb_opt_rmse:.4f}")
print(f"Test R2   : {xgb_opt_r2:.4f}")

In [ ]:
# -- Visualisation Optuna XGB --
trials_xgb  = [t.value for t in study_xgb.trials]
best_xgb_cm = np.minimum.accumulate(trials_xgb)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(range(len(trials_xgb)), trials_xgb,
                alpha=0.5, color="seagreen", s=25, label="MAE trial")
axes[0].plot(range(len(best_xgb_cm)), best_xgb_cm,
             color="red", linewidth=2, label="Meilleur cumulatif")
axes[0].axhline(study_xgb.best_value, color="orange", linestyle="--", alpha=0.7,
                label=f"Best = {study_xgb.best_value:.3f}")
axes[0].set_title("Optuna XGBoost — Historique d'optimisation")
axes[0].set_xlabel("Trial #")
axes[0].set_ylabel("MAE (minutes)")
axes[0].legend()
axes[0].grid(alpha=0.3)

df_t_xgb = study_xgb.trials_dataframe()
param_cols_xgb = [c for c in df_t_xgb.columns if c.startswith("params_")]
corrs_xgb = {}
for col in param_cols_xgb:
    try:
        c = df_t_xgb[col].astype(float).corr(df_t_xgb["value"])
        corrs_xgb[col.replace("params_", "")] = abs(c) if not np.isnan(c) else 0
    except Exception:
        pass
corrs_xgb_s = dict(sorted(corrs_xgb.items(), key=lambda x: x[1], reverse=True))
axes[1].barh(list(corrs_xgb_s.keys()), list(corrs_xgb_s.values()),
             color="seagreen", edgecolor="black")
axes[1].set_title("XGB — Importance des hyperparametres")
axes[1].set_xlabel("|Correlation|")
axes[1].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

// ==================================================================================================
// SECTION 6 — COMPARAISON FINALE : BASELINE / GRIDSEARCHCV / OPTUNA
// ==================================================================================================

In [ ]:
comparison = pd.DataFrame({
    "Modele": [
        "RF Baseline (NB04)",
        "RF GridSearchCV",
        "RF Optuna",
        "XGBoost Baseline (NB04)",
        "XGBoost GridSearchCV",
        "XGBoost Optuna",
    ],
    "MAE": [
        RF_BASE_MAE,  rf_gs_mae,  rf_opt_mae,
        XGB_BASE_MAE, xgb_gs_mae, xgb_opt_mae,
    ],
    "RMSE": [
        RF_BASE_RMSE,  rf_gs_rmse,  rf_opt_rmse,
        XGB_BASE_RMSE, xgb_gs_rmse, xgb_opt_rmse,
    ],
    "Methode": [
        "Baseline", "GridSearchCV", "Optuna",
        "Baseline", "GridSearchCV", "Optuna",
    ]
})
comparison["MAE"]  = comparison["MAE"].round(3)
comparison["RMSE"] = comparison["RMSE"].round(3)
print(comparison.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
method_colors = ["grey", "steelblue", "darkorange"]
method_labels = ["Baseline", "GridSearchCV", "Optuna"]

rf_maes  = [RF_BASE_MAE,  rf_gs_mae,  rf_opt_mae]
xgb_maes = [XGB_BASE_MAE, xgb_gs_mae, xgb_opt_mae]

for ax, maes, title, base_mae in [
    (axes[0], rf_maes,  "Random Forest",  RF_BASE_MAE),
    (axes[1], xgb_maes, "XGBoost",        XGB_BASE_MAE)
]:
    bars = ax.bar(method_labels, maes,
                  color=method_colors, edgecolor="black", width=0.5)
    for b, v in zip(bars, maes):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.04,
                f"{v:.3f}", ha="center", fontweight="bold", fontsize=10)
    ax.axhline(base_mae, color="red", linestyle="--", alpha=0.6,
               label=f"Baseline = {base_mae}")
    ax.set_title(f"{title} — Impact de l'optimisation")
    ax.set_ylabel("MAE (minutes, 50k sample)")
    ax.set_ylim(0, max(maes) * 1.3)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

ml_p  = mpatches.Patch(color="grey",       label="Baseline")
gs_p  = mpatches.Patch(color="steelblue",  label="GridSearchCV")
opt_p = mpatches.Patch(color="darkorange", label="Optuna")
fig.legend(handles=[ml_p, gs_p, opt_p],
           loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.03))
plt.suptitle("Baseline vs GridSearchCV vs Optuna — MAE (taxi_out, 50k lignes)")
plt.tight_layout()
plt.show()

In [ ]:
# -- GridSearchCV vs Optuna : comparaison qualitative --
gs_vs_opt = pd.DataFrame({
    "Critere": [
        "Type d'exploration",
        "Espace de recherche",
        "Nombre d'evaluations",
        "Parallelisable",
        "Pruning des mauvais runs",
        "Reproductibilite",
        "Facilite d'utilisation",
        "Meilleur MAE RF",
        "Meilleur MAE XGB",
        "Temps RF",
        "Temps XGB",
    ],
    "GridSearchCV": [
        "Exhaustive (grille fixe)",
        "Discret uniquement",
        f"{total_rf} fits  (cv=3)",
        "Oui (n_jobs=-1)",
        "Non",
        "Totale",
        "Tres simple",
        f"{rf_gs_mae:.3f}",
        f"{xgb_gs_mae:.3f}",
        f"{grid_rf_time:.0f}s",
        f"{grid_xgb_time:.0f}s",
    ],
    "Optuna (TPE)": [
        "Bayesienne intelligente",
        "Continu + discret + log",
        f"{N_TRIALS} trials",
        "Oui (n_jobs dans objective)",
        "Oui (MedianPruner)",
        "Seed fixable",
        "Moderee",
        f"{rf_opt_mae:.3f}",
        f"{xgb_opt_mae:.3f}",
        f"{optuna_rf_time:.0f}s",
        f"{optuna_xgb_time:.0f}s",
    ]
}).set_index("Critere")

print(gs_vs_opt.to_string())

// ==================================================================================================
// SECTION 7 — BONNES PRATIQUES ET RECOMMANDATIONS
// ==================================================================================================
//
//  EARLYSTOPPING :
//    - Toujours utiliser restore_best_weights=True
//    - Combiner avec ReduceLROnPlateau (patience_lr < patience_es)
//    - patience recommandee : 7 pour MLP, 5 pour LSTM
//    - min_delta=0.001 evite les arrets sur des micro-ameliorations
//
//  GRIDSEARCHCV :
//    - Ideal pour petites grilles (< 30 combinaisons x cv)
//    - Utiliser negative_mae comme scoring pour sklearn
//    - Analyse des resultats via cv_results_ DataFrame
//
//  OPTUNA :
//    - Preferer pour grands espaces continus (learning_rate, reg_alpha...)
//    - Fixer seed via TPESampler(seed=42) pour reproductibilite
//    - Augmenter n_trials si le budget le permet (100+ pour production)
//    - Ajouter pruning : optuna.pruners.MedianPruner()
//
// ==================================================================================================

In [ ]:
print("=" * 70)
print("RESULTATS FINAUX — NOTEBOOK 10 : CONTROLE DE L'ENTRAINEMENT")
print("=" * 70)

print("\n1. EARLYSTOPPING MLP :")
print(f"   {'Patience':<12} {'Epochs':>8} {'MAE':>8} {'RMSE':>9}")
for p in [3, 7, 15]:
    r = results_es_mlp[p]
    print(f"   {p:<12} {r['epochs']:>8} {r['mae']:>8.3f} {r['rmse']:>9.3f}")
print(f"   => patience recommandee : {best_p_mlp}")

print("\n2. EARLYSTOPPING LSTM :")
print(f"   {'Patience':<12} {'Epochs':>8} {'MAE':>8} {'R2':>8}")
for p in [3, 5, 10]:
    r = results_es_lstm[p]
    print(f"   {p:<12} {r['epochs']:>8} {r['mae']:>8.4f} {r['r2']:>8.4f}")
print(f"   => patience recommandee : {best_p_lstm}")

print("\n3. GRIDSEARCHCV (50k lignes) :")
print(f"   RF  baseline {RF_BASE_MAE:.3f}  ->  tuned {rf_gs_mae:.3f}  "
      f"(delta = {RF_BASE_MAE - rf_gs_mae:+.3f}  en {grid_rf_time:.0f}s)")
print(f"   XGB baseline {XGB_BASE_MAE:.3f}  ->  tuned {xgb_gs_mae:.3f}  "
      f"(delta = {XGB_BASE_MAE - xgb_gs_mae:+.3f}  en {grid_xgb_time:.0f}s)")

print("\n4. OPTUNA TPE (50 trials, 50k lignes) :")
print(f"   RF  baseline {RF_BASE_MAE:.3f}  ->  tuned {rf_opt_mae:.3f}  "
      f"(delta = {RF_BASE_MAE - rf_opt_mae:+.3f}  en {optuna_rf_time:.0f}s)")
print(f"   XGB baseline {XGB_BASE_MAE:.3f}  ->  tuned {xgb_opt_mae:.3f}  "
      f"(delta = {XGB_BASE_MAE - xgb_opt_mae:+.3f}  en {optuna_xgb_time:.0f}s)")

print("=" * 70)